### **1) Imports and connection**

In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

# Connect to the SQLite database
DB_PATH = Path("../data/raw/database.sqlite")
conn = sqlite3.connect(DB_PATH)

print(f"Connected to: {DB_PATH.resolve()}")
print(f"DB file size: {DB_PATH.stat().st_size / 1024 / 1024:.1f} MB")

Connected to: /home/muhammed786fiyas/Desktop/Projects/ml_ops/final_project/mlops_end_to_end_project/data/raw/database.sqlite
DB file size: 298.6 MB


### **2) Data Exploration**

#### List all tables

In [3]:
# What tables exist in the database?
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'", 
    conn
)
print(tables)

                name
0    sqlite_sequence
1  Player_Attributes
2             Player
3              Match
4             League
5            Country
6               Team
7    Team_Attributes


#### Row counts per table


In [4]:
# How big is each table?
for table_name in tables['name']:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table_name}", conn).iloc[0, 0]
    print(f"{table_name:25s} {count:>10,} rows")

sqlite_sequence                    7 rows
Player_Attributes            183,978 rows
Player                        11,060 rows
Match                         25,979 rows
League                            11 rows
Country                           11 rows
Team                             299 rows
Team_Attributes                1,458 rows


#### Peek at the Match table

In [5]:
# The Match table is our heart — every row is one match outcome
match = pd.read_sql_query("SELECT * FROM Match LIMIT 5", conn)
print(f"Match table has {len(match.columns)} columns")
print(match.columns.tolist())

Match table has 115 columns
['id', 'country_id', 'league_id', 'season', 'stage', 'date', 'match_api_id', 'home_team_api_id', 'away_team_api_id', 'home_team_goal', 'away_team_goal', 'home_player_X1', 'home_player_X2', 'home_player_X3', 'home_player_X4', 'home_player_X5', 'home_player_X6', 'home_player_X7', 'home_player_X8', 'home_player_X9', 'home_player_X10', 'home_player_X11', 'away_player_X1', 'away_player_X2', 'away_player_X3', 'away_player_X4', 'away_player_X5', 'away_player_X6', 'away_player_X7', 'away_player_X8', 'away_player_X9', 'away_player_X10', 'away_player_X11', 'home_player_Y1', 'home_player_Y2', 'home_player_Y3', 'home_player_Y4', 'home_player_Y5', 'home_player_Y6', 'home_player_Y7', 'home_player_Y8', 'home_player_Y9', 'home_player_Y10', 'home_player_Y11', 'away_player_Y1', 'away_player_Y2', 'away_player_Y3', 'away_player_Y4', 'away_player_Y5', 'away_player_Y6', 'away_player_Y7', 'away_player_Y8', 'away_player_Y9', 'away_player_Y10', 'away_player_Y11', 'home_player_1', 'h

#### Look at columns we actually care about


In [6]:
# The columns we'll likely use
cols_of_interest = [
    'id', 'country_id', 'league_id', 'season', 'stage', 'date',
    'home_team_api_id', 'away_team_api_id',
    'home_team_goal', 'away_team_goal'
]
match_clean = pd.read_sql_query(
    f"SELECT {', '.join(cols_of_interest)} FROM Match LIMIT 10", 
    conn
)
match_clean

,id,country_id,league_id,season,stage,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,9987,9993,1,1
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,10000,9994,0,0
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,9984,8635,0,3
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,9991,9998,5,0
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,7947,9985,1,3
5,6,1,1,2008/2009,1,2008-09-24 00:00:00,8203,8342,1,1
6,7,1,1,2008/2009,1,2008-08-16 00:00:00,9999,8571,2,2
7,8,1,1,2008/2009,1,2008-08-16 00:00:00,4049,9996,1,2
8,9,1,1,2008/2009,1,2008-08-16 00:00:00,10001,9986,1,0
9,10,1,1,2008/2009,10,2008-11-01 00:00:00,8342,8571,4,1


####  Compute the target variable distribution

In [7]:
# Our target: Home Win (H) / Draw (D) / Away Win (A)
all_matches = pd.read_sql_query(
    "SELECT home_team_goal, away_team_goal FROM Match",
    conn
)

import numpy as np
def get_outcome(row):
    if row['home_team_goal'] > row['away_team_goal']:
        return 'H'
    elif row['home_team_goal'] < row['away_team_goal']:
        return 'A'
    else:
        return 'D'

all_matches['outcome'] = all_matches.apply(get_outcome, axis=1)
print(all_matches['outcome'].value_counts(normalize=True).round(3))
print(f"\nTotal matches: {len(all_matches):,}")

# Also show class counts
print("\nRaw counts:")
print(all_matches['outcome'].value_counts())

outcome
H    0.459
A    0.287
D    0.254
Name: proportion, dtype: float64

Total matches: 25,979

Raw counts:
outcome
H    11917
A     7466
D     6596
Name: count, dtype: int64


#### Look at the Team table

In [8]:
team = pd.read_sql_query("SELECT * FROM Team LIMIT 10", conn)
print("Team columns:", team.columns.tolist())
team

Team columns: ['id', 'team_api_id', 'team_fifa_api_id', 'team_long_name', 'team_short_name']


,id,team_api_id,team_fifa_api_id,team_long_name,team_short_name
0,1,9987,673.0,KRC Genk,GEN
1,2,9993,675.0,Beerschot AC,BAC
2,3,10000,15005.0,SV Zulte-Waregem,ZUL
3,4,9994,2007.0,Sporting Lokeren,LOK
4,5,9984,1750.0,KSV Cercle Brugge,CEB
5,6,8635,229.0,RSC Anderlecht,AND
6,7,9991,674.0,KAA Gent,GEN
7,8,9998,1747.0,RAEC Mons,MON
8,9,7947,NaN,FCV Dender EH,DEN
9,10,9985,232.0,Standard de Liège,STL


####  Look at Team_Attributes (FIFA ratings)

In [9]:
team_attrs = pd.read_sql_query("SELECT * FROM Team_Attributes LIMIT 5", conn)
print(f"Team_Attributes has {len(team_attrs.columns)} columns")
print(team_attrs.columns.tolist())
team_attrs.head()

Team_Attributes has 25 columns
['id', 'team_fifa_api_id', 'team_api_id', 'date', 'buildUpPlaySpeed', 'buildUpPlaySpeedClass', 'buildUpPlayDribbling', 'buildUpPlayDribblingClass', 'buildUpPlayPassing', 'buildUpPlayPassingClass', 'buildUpPlayPositioningClass', 'chanceCreationPassing', 'chanceCreationPassingClass', 'chanceCreationCrossing', 'chanceCreationCrossingClass', 'chanceCreationShooting', 'chanceCreationShootingClass', 'chanceCreationPositioningClass', 'defencePressure', 'defencePressureClass', 'defenceAggression', 'defenceAggressionClass', 'defenceTeamWidth', 'defenceTeamWidthClass', 'defenceDefenderLineClass']


,id,team_fifa_api_id,team_api_id,date,buildUpPlaySpeed,buildUpPlaySpeedClass,buildUpPlayDribbling,buildUpPlayDribblingClass,buildUpPlayPassing,buildUpPlayPassingClass,...,chanceCreationShooting,chanceCreationShootingClass,chanceCreationPositioningClass,defencePressure,defencePressureClass,defenceAggression,defenceAggressionClass,defenceTeamWidth,defenceTeamWidthClass,defenceDefenderLineClass
0,1,434,9930,2010-02-22 00:00:00,60,Balanced,NaN,Little,50,Mixed,...,55,Normal,Organised,50,Medium,55,Press,45,Normal,Cover
1,2,434,9930,2014-09-19 00:00:00,52,Balanced,48.0,Normal,56,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
2,3,434,9930,2015-09-10 00:00:00,47,Balanced,41.0,Normal,54,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
3,4,77,8485,2010-02-22 00:00:00,70,Fast,NaN,Little,70,Long,...,70,Lots,Organised,60,Medium,70,Double,70,Wide,Cover
4,5,77,8485,2011-02-22 00:00:00,47,Balanced,NaN,Little,52,Mixed,...,52,Normal,Organised,47,Medium,47,Press,52,Normal,Cover


#### Date range and league coverage

In [10]:
# What time range and which leagues?
date_range = pd.read_sql_query(
    "SELECT MIN(date) AS min_date, MAX(date) AS max_date FROM Match", 
    conn
)
print("Date range:", date_range.iloc[0].to_dict())

leagues = pd.read_sql_query(
    """
    SELECT l.name AS league, COUNT(*) AS match_count
    FROM Match m JOIN League l ON m.league_id = l.id
    GROUP BY l.name
    ORDER BY match_count DESC
    """,
    conn
)
print("\nLeagues:")
print(leagues)

Date range: {'min_date': '2008-07-18 00:00:00', 'max_date': '2016-05-25 00:00:00'}

Leagues:
                      league  match_count
0            Spain LIGA BBVA         3040
1             France Ligue 1         3040
2     England Premier League         3040
3              Italy Serie A         3017
4     Netherlands Eredivisie         2448
5      Germany 1. Bundesliga         2448
6   Portugal Liga ZON Sagres         2052
7         Poland Ekstraklasa         1920
8    Scotland Premier League         1824
9     Belgium Jupiler League         1728
10  Switzerland Super League         1422


In [11]:
conn.close()
print("Done exploring.")

Done exploring.
